In [ ]:
# Challenge 2 — Imports
# Only what this challenge needs: HTTP calls, ADK agent/runner/session,
# message types, and unit-test scaffolding.
import os
import unittest
from typing import Any, Dict
from unittest import mock

import requests

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

print("Imports ready.")


In [ ]:
import os
import google.auth
from google import genai

# ---- Maps key: the only API key this notebook needs -------------------------
def get_secret(name: str, prompt: str) -> str:
    """Read a credential from the environment, else prompt via input().

    getpass() hangs on some VS Code kernels, so we use input(). Nothing is
    hardcoded; set the env var beforehand to skip the prompt.
    """
    value = os.environ.get(name, "").strip()
    if value:
        print("{}: loaded from environment ({} chars).".format(name, len(value)))
        return value
    value = input(prompt).strip()
    os.environ[name] = value
    return value


GOOGLE_MAPS_API_KEY = get_secret(
    "GOOGLE_MAPS_API_KEY", "Enter your Google Maps Geocoding API key: ")
assert GOOGLE_MAPS_API_KEY, "Google Maps API key is required for geocoding."

# ---- Model auth: Vertex AI via this kernel's ambient lab credentials --------
_creds, _adc_project = google.auth.default()
GOOGLE_CLOUD_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip() or _adc_project
GOOGLE_CLOUD_LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "").strip() or "us-central1"

# google-genai only accepts "1" or "true" here and defaults to "0"; leaving it
# unset is what sends the client down the API-key path and raises
# "No API key was provided."
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = GOOGLE_CLOUD_LOCATION
# A Maps key here would shadow Vertex auth -> 403 API_KEY_SERVICE_BLOCKED.
os.environ.pop("GOOGLE_API_KEY", None)
os.environ.pop("GEMINI_API_KEY", None)

# Prove the flag took effect before ADK ever builds its own client.
_probe_client = genai.Client()
assert _probe_client.vertexai, "Vertex mode did not engage; check the env vars above."
print("Vertex AI engaged: project={} location={}".format(
    GOOGLE_CLOUD_PROJECT, GOOGLE_CLOUD_LOCATION))

_probe = _probe_client.models.generate_content(
    model="gemini-2.5-flash", contents="Reply with the single word: ready")
print("Model auth OK ->", (_probe.text or "").strip())


In [ ]:
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"

NWS_API_BASE = "https://api.weather.gov"
# The NWS API mandates a User-Agent header; omitting it returns 403 Forbidden.
NWS_USER_AGENT = "(challenge1-weather-agent, nathaniel.morrow@example.com)"

GEOCODE_API_URL = "https://maps.googleapis.com/maps/api/geocode/json"
REQUEST_TIMEOUT_SECONDS = 15

APP_NAME = "weather_app"
USER_ID = "workshop-user"


In [ ]:
def get_location_lat_long(city: str, state: str) -> Dict[str, Any]:
    """Convert a US city and state into latitude and longitude coordinates.

    Call this FIRST whenever the user names a place instead of giving coordinates.
    Pass the returned latitude and longitude to `get_current_weather`.

    Args:
        city: The city name, for example "Denver" or "Miami".
        state: The state name or two-letter abbreviation, for example "CO" or "Florida".

    Returns:
        On success: {"status": "success", "formatted_address": str,
                     "latitude": float, "longitude": float}
        On failure: {"status": "error", "error_message": str}

        Failure cases include an unrecognizable location and any location outside
        the United States, since the National Weather Service only covers the US.
    """
    if not GOOGLE_MAPS_API_KEY:
        return {"status": "error", "error_message": "No Google Maps API key configured."}

    params = {"address": "{}, {}".format(city, state), "key": GOOGLE_MAPS_API_KEY}
    try:
        response = requests.get(GEOCODE_API_URL, params=params,
                                timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
        payload = response.json()
    except requests.exceptions.RequestException as exc:
        return {"status": "error",
                "error_message": "Geocoding request failed: {}".format(exc)}

    api_status = payload.get("status")
    results = payload.get("results") or []
    if api_status != "OK" or not results:
        return {"status": "error",
                "error_message": "Could not geocode '{}, {}' (Geocoding API status: {}).".format(
                    city, state, api_status)}

    top = results[0]

    # Reject non-US locations early so we never waste a call on an out-of-bounds NWS point.
    country_code = None
    for component in top.get("address_components", []):
        if "country" in component.get("types", []):
            country_code = component.get("short_name")
            break

    if country_code and country_code != "US":
        return {"status": "error",
                "error_message": ("'{}' resolved to {}, which is outside the United States. "
                                  "The National Weather Service only covers the US and its "
                                  "territories.".format(top.get("formatted_address"), country_code))}

    location = top["geometry"]["location"]
    return {
        "status": "success",
        "formatted_address": top.get("formatted_address"),
        "latitude": location["lat"],
        "longitude": location["lng"],
    }


print(get_location_lat_long("Denver", "CO"))


In [ ]:
def get_current_weather(latitude: float, longitude: float) -> Dict[str, Any]:
    """Get the real-time weather forecast for US coordinates from the National Weather Service.

    Requires latitude/longitude, so call `get_location_lat_long` first if the user
    gave you a city name rather than coordinates.

    This performs the NWS two-step flow:
      1. GET /points/{lat},{lon} -> returns grid METADATA, not weather. Contains a
         dynamically generated `properties.forecast` URL for that grid square.
      2. GET that forecast URL -> returns `properties.periods[]`; index 0 is the
         current/most immediate period.

    Args:
        latitude: Latitude in decimal degrees, for example 39.7392.
        longitude: Longitude in decimal degrees, for example -104.9903.

    Returns:
        On success: {"status": "success", "period": str, "temperature": int,
                     "temperature_unit": str, "short_forecast": str,
                     "detailed_forecast": str, "summary": str}
        On failure: {"status": "error", "error_message": str}

        A 404 from the NWS means the point is outside US coverage.
    """
    headers = {"User-Agent": NWS_USER_AGENT, "Accept": "application/geo+json"}
    points_url = "{}/points/{},{}".format(NWS_API_BASE, latitude, longitude)

    try:
        # Hop 1: metadata lookup.
        points_response = requests.get(points_url, headers=headers,
                                       timeout=REQUEST_TIMEOUT_SECONDS)
        if points_response.status_code == 404:
            return {"status": "error",
                    "error_message": ("The National Weather Service has no data for {},{}. "
                                      "This location is most likely outside the United "
                                      "States.".format(latitude, longitude))}
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]

        # Hop 2: the actual forecast.
        forecast_response = requests.get(forecast_url, headers=headers,
                                         timeout=REQUEST_TIMEOUT_SECONDS)
        forecast_response.raise_for_status()
        periods = forecast_response.json()["properties"]["periods"]
    except requests.exceptions.RequestException as exc:
        return {"status": "error", "error_message": "NWS request failed: {}".format(exc)}
    except (KeyError, IndexError, ValueError) as exc:
        return {"status": "error",
                "error_message": "Unexpected NWS response shape: {}".format(exc)}

    if not periods:
        return {"status": "error", "error_message": "NWS returned an empty forecast."}

    current = periods[0]
    return {
        "status": "success",
        "period": current.get("name"),
        "temperature": current.get("temperature"),
        "temperature_unit": current.get("temperatureUnit"),
        "short_forecast": current.get("shortForecast"),
        "detailed_forecast": current.get("detailedForecast"),
        "summary": "{}: {} Temperature near {}\u00b0{}. {}".format(
            current.get("name"), current.get("shortForecast"),
            current.get("temperature"), current.get("temperatureUnit"),
            current.get("detailedForecast")),
    }


# Direct proof the tools chain: Denver, CO -> coordinates -> live forecast.
_loc = get_location_lat_long("Denver", "CO")
print(get_current_weather(_loc["latitude"], _loc["longitude"])["summary"])


In [ ]:
weather_agent = Agent(
    name="weather_agent",
    model=MODEL_GEMINI_2_5_FLASH,
    description="Reports real-time US weather using the National Weather Service API.",
    instruction=(
        "You are a helpful weather assistant for locations in the United States.\n"
        "\n"
        "To answer a weather question you must chain two tools:\n"
        "1. Call `get_location_lat_long` with the city and state to get coordinates.\n"
        "2. Pass those coordinates to `get_current_weather` to get the forecast.\n"
        "\n"
        "Then write a short, friendly weather summary in plain prose: mention the "
        "period, the conditions, and the temperature, and add any notable detail "
        "such as rain chances, wind, or heat index.\n"
        "\n"
        "Rules:\n"
        "- If a tool returns status 'error', explain the problem politely in your own "
        "  words. Never invent weather data.\n"
        "- You can only cover the United States and its territories. For anywhere else, "
        "  say so plainly and do not guess.\n"
        "- If the user asks about something other than weather, politely decline and "
        "  say you only handle US weather.\n"
        "- If the user names a city without a state, use the most well-known US city "
        "  with that name."
    ),
    tools=[get_location_lat_long, get_current_weather],
)

session_service = InMemorySessionService()
runner = Runner(app_name=APP_NAME, agent=weather_agent, session_service=session_service)

print("Agent '{}' ready on model '{}' with {} tools.".format(
    weather_agent.name, MODEL_GEMINI_2_5_FLASH, len(weather_agent.tools)))


In [ ]:
async def ask_weather_agent_async(prompt: str, session_id: str,
                                  show_tool_calls: bool = True) -> str:
    """Send one prompt to the weather agent and return its final text response.

    Takes an explicit session_id so callers can either reuse a session across
    turns (preserving conversation history) or pass a fresh one per question.

    Args:
        prompt: The user's question, e.g. "What's the weather in Denver, CO?".
        session_id: An existing ADK session id.
        show_tool_calls: If True, print each tool call and result status so the
            agent's two-step chaining (geocode -> forecast) is visible.

    Returns:
        The agent's final response text, or "(no response)" if none was produced.
    """
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    final_text = "(no response)"

    async for event in runner.run_async(user_id=USER_ID, session_id=session_id,
                                        new_message=message):
        if show_tool_calls:
            for call in event.get_function_calls():
                print("   [tool call] {}({})".format(call.name, dict(call.args)))
            for resp in event.get_function_responses():
                status = (resp.response or {}).get("status", "?")
                print("   [tool result] {} -> {}".format(resp.name, status))

        # The final response carries the agent's prose summary; earlier events
        # carry the intermediate function calls and their results.
        if event.is_final_response() and event.content and event.content.parts:
            texts = [p.text for p in event.content.parts if p.text]
            if texts:
                final_text = "".join(texts).strip()

    return final_text


async def ask_weather_agent(prompt: str, show_tool_calls: bool = True) -> str:
    """Convenience wrapper: ask one question in its own fresh session.

    Each call starts clean, so test prompts cannot influence one another.
    """
    session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID)
    return await ask_weather_agent_async(prompt, session.id, show_tool_calls)


# Requirement 3: demonstrate the agent can be invoked.
# Top-level await works directly in Jupyter/Colab notebook cells.
print("User: What's the weather in Denver, Colorado?\n")
print("Agent:", await ask_weather_agent("What's the weather in Denver, Colorado?"))


In [ ]:
# Requirement 5: prove it works across multiple US cities.
US_CITIES = [
    "What's the weather in Miami, FL?",
    "How's the weather in Chicago, Illinois?",
    "Give me the forecast for Denver, CO.",
    "What's it like in Seattle, Washington right now?",
    "Weather in Phoenix, AZ?",
]

for prompt in US_CITIES:
    answer = await ask_weather_agent(prompt, show_tool_calls=False)
    print("=" * 78)
    print("User:  {}".format(prompt))
    print("Agent: {}\n".format(answer))


In [ ]:
# Guardrails: the agent should decline gracefully rather than hallucinate.
GUARDRAIL_PROMPTS = [
    ("Non-US city",    "What's the weather in London, England?"),
    ("Nonsense place", "What's the weather in Fakeburg, Zanzibaria?"),
    ("Off-topic",      "Who voiced Donald Duck?"),
]

for label, prompt in GUARDRAIL_PROMPTS:
    answer = await ask_weather_agent(prompt, show_tool_calls=False)
    print("=" * 78)
    print("[{}]".format(label))
    print("User:  {}".format(prompt))
    print("Agent: {}\n".format(answer))


In [ ]:
# Requirement 6: unit tests against the tool functions directly.
# HTTP is mocked so these are fast, offline, and deterministic — no live API calls,
# and no real API key needed.

def _fake_response(status_code=200, payload=None):
    resp = mock.Mock()
    resp.status_code = status_code
    resp.json.return_value = payload or {}
    if status_code >= 400:
        resp.raise_for_status.side_effect = requests.exceptions.HTTPError(
            "{} error".format(status_code))
    else:
        resp.raise_for_status.return_value = None
    return resp


GEOCODE_OK = {
    "status": "OK",
    "results": [{
        "formatted_address": "Denver, CO, USA",
        "address_components": [{"types": ["country", "political"], "short_name": "US"}],
        "geometry": {"location": {"lat": 39.7392358, "lng": -104.990251}},
    }],
}
GEOCODE_LONDON = {
    "status": "OK",
    "results": [{
        "formatted_address": "London, UK",
        "address_components": [{"types": ["country", "political"], "short_name": "GB"}],
        "geometry": {"location": {"lat": 51.5072178, "lng": -0.1275862}},
    }],
}
POINTS_OK = {"properties": {"forecast": "https://api.weather.gov/gridpoints/BOU/62,61/forecast"}}
FORECAST_OK = {"properties": {"periods": [
    {"name": "Today", "temperature": 88, "temperatureUnit": "F",
     "shortForecast": "Sunny", "detailedForecast": "Sunny, with a high near 88."},
    {"name": "Tonight", "temperature": 60, "temperatureUnit": "F"},
]}}


class TestGeocodingTool(unittest.TestCase):
    @mock.patch("requests.get")
    def test_returns_coordinates_for_us_city(self, mock_get):
        mock_get.return_value = _fake_response(200, GEOCODE_OK)
        result = get_location_lat_long("Denver", "CO")
        self.assertEqual(result["status"], "success")
        self.assertAlmostEqual(result["latitude"], 39.7392358)
        self.assertAlmostEqual(result["longitude"], -104.990251)

    @mock.patch("requests.get")
    def test_api_key_is_passed_as_param(self, mock_get):
        mock_get.return_value = _fake_response(200, GEOCODE_OK)
        get_location_lat_long("Denver", "CO")
        _, kwargs = mock_get.call_args
        self.assertEqual(kwargs["params"]["key"], GOOGLE_MAPS_API_KEY)
        self.assertEqual(kwargs["params"]["address"], "Denver, CO")

    @mock.patch("requests.get")
    def test_zero_results_is_an_error(self, mock_get):
        mock_get.return_value = _fake_response(200, {"status": "ZERO_RESULTS", "results": []})
        result = get_location_lat_long("Fakeburg", "Zanzibaria")
        self.assertEqual(result["status"], "error")
        self.assertIn("ZERO_RESULTS", result["error_message"])

    @mock.patch("requests.get")
    def test_non_us_location_is_rejected(self, mock_get):
        mock_get.return_value = _fake_response(200, GEOCODE_LONDON)
        result = get_location_lat_long("London", "England")
        self.assertEqual(result["status"], "error")
        self.assertIn("outside the United States", result["error_message"])

    @mock.patch("requests.get", side_effect=requests.exceptions.ConnectionError("boom"))
    def test_network_failure_is_handled(self, mock_get):
        result = get_location_lat_long("Denver", "CO")
        self.assertEqual(result["status"], "error")
        self.assertIn("Geocoding request failed", result["error_message"])


class TestWeatherTool(unittest.TestCase):
    @mock.patch("requests.get")
    def test_two_hop_flow_and_summary(self, mock_get):
        mock_get.side_effect = [_fake_response(200, POINTS_OK),
                                _fake_response(200, FORECAST_OK)]
        result = get_current_weather(39.7392, -104.9903)
        self.assertEqual(result["status"], "success")
        self.assertEqual(mock_get.call_count, 2)
        self.assertEqual(mock_get.call_args_list[0][0][0],
                         "https://api.weather.gov/points/39.7392,-104.9903")
        self.assertEqual(mock_get.call_args_list[1][0][0],
                         POINTS_OK["properties"]["forecast"])
        self.assertEqual(result["period"], "Today")      # periods[0], not periods[1]
        self.assertEqual(result["temperature"], 88)
        self.assertIn("Sunny", result["summary"])

    @mock.patch("requests.get")
    def test_user_agent_header_always_sent(self, mock_get):
        mock_get.side_effect = [_fake_response(200, POINTS_OK),
                                _fake_response(200, FORECAST_OK)]
        get_current_weather(39.7392, -104.9903)
        for call in mock_get.call_args_list:
            self.assertTrue(call[1]["headers"].get("User-Agent"))

    @mock.patch("requests.get")
    def test_404_reported_as_outside_us(self, mock_get):
        mock_get.return_value = _fake_response(404, {"title": "Data Unavailable"})
        result = get_current_weather(51.5072, -0.1276)
        self.assertEqual(result["status"], "error")
        self.assertIn("outside the United States", result["error_message"])

    @mock.patch("requests.get")
    def test_server_error_is_handled(self, mock_get):
        mock_get.return_value = _fake_response(500)
        result = get_current_weather(39.7392, -104.9903)
        self.assertEqual(result["status"], "error")
        self.assertIn("NWS request failed", result["error_message"])

    @mock.patch("requests.get")
    def test_empty_periods_is_handled(self, mock_get):
        mock_get.side_effect = [_fake_response(200, POINTS_OK),
                                _fake_response(200, {"properties": {"periods": []}})]
        result = get_current_weather(39.7392, -104.9903)
        self.assertEqual(result["status"], "error")
        self.assertIn("empty forecast", result["error_message"])

    @mock.patch("requests.get")
    def test_malformed_payload_is_handled(self, mock_get):
        mock_get.return_value = _fake_response(200, {"unexpected": True})
        result = get_current_weather(39.7392, -104.9903)
        self.assertEqual(result["status"], "error")
        self.assertIn("Unexpected NWS response shape", result["error_message"])


_suite = unittest.TestSuite([
    unittest.TestLoader().loadTestsFromTestCase(TestGeocodingTool),
    unittest.TestLoader().loadTestsFromTestCase(TestWeatherTool),
])
unittest.TextTestRunner(verbosity=2).run(_suite)


In [ ]:
# Manual / interactive testing: ask the agent anything, on demand.
# Type a question, or press Enter on an empty line to stop.
#
# Try things like:
#   Denver, CO                          -> bare "City, ST" works
#   What's the weather in Austin?       -> no state; agent picks the well-known city
#   Will it rain in Tampa, FL today?    -> follow-up phrasing
#   London, England                     -> should decline (non-US)
#   Who won the World Series?           -> should decline (off-topic)

SHOW_TOOL_CALLS = True   # set False for a cleaner transcript

print("Interactive weather agent. Empty line to quit.\n")

while True:
    try:
        question = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nStopped.")
        break

    if not question:
        print("Done.")
        break

    try:
        answer = await ask_weather_agent(question, show_tool_calls=SHOW_TOOL_CALLS)
        print("Agent: {}\n".format(answer))
    except Exception as exc:
        # Keep the loop alive so one bad call doesn't end the session.
        print("Agent call failed: {}: {}\n".format(type(exc).__name__, str(exc)[:300]))
